# Phase 3: Alpha Research & Statistical Learning  
## Notebook 03_06 — ARIMA vs LSTM Benchmark

### Research question

When forecasting financial time series, how do classical ARIMA-style models compare with an LSTM sequence model under a fair, leakage-controlled benchmark?

This notebook follows naturally from:

```text
03_01_garch_volatility_modeling
03_02_har_realized_volatility_forecasting
03_05_vector_autoregression_granger
```

The goal is not to prove that deep learning is better or worse. The goal is to build a benchmark where:

1. the data split respects time;
2. all scalers are fit on training data only;
3. baselines are included;
4. ARIMA gets a fair stationary target;
5. LSTM gets a fair sequence-window setup;
6. results are evaluated out of sample;
7. forecasting performance is translated into a toy trading/risk signal carefully.

It covers:

1. synthetic financial time-series generation;
2. stationarity and transformation choices;
3. train/validation/test split;
4. naïve and historical-mean baselines;
5. AR/ARIMA model selection;
6. walk-forward ARIMA-style forecasting;
7. sequence-window feature construction;
8. LSTM model training if TensorFlow is available;
9. neural fallback benchmark if TensorFlow is unavailable;
10. forecast evaluation metrics;
11. directional accuracy;
12. Diebold-Mariano-style paired loss comparison;
13. toy trading signal from forecasts;
14. leakage checklist;
15. saved benchmark reports and manifest.

The key idea:

> A model comparison is only meaningful if the benchmark protocol is fair. In finance, leakage and weak baselines make many ML wins fake.

## 1. Why ARIMA vs LSTM?

ARIMA is a classical statistical time-series model.

It works well when:

- the target is stationary;
- dynamics are mostly linear;
- the signal-to-noise ratio is low;
- interpretability matters;
- data is limited.

LSTM is a neural sequence model.

It can help when:

- nonlinear temporal patterns exist;
- long-range dependencies matter;
- there is enough data;
- features are well-engineered;
- regularisation and validation are done properly.

But financial returns are noisy.

A large model can easily overfit.

This notebook therefore benchmarks ARIMA and LSTM against simple baselines, rather than treating model sophistication as evidence of quality.

## 2. Forecast target

We distinguish between:

### Price level

$$
P_t
$$

Price levels are usually non-stationary.

### Log return

$$
r_t = \log P_t - \log P_{t-1}
$$

Returns are usually closer to stationary.

This notebook forecasts next-day return:

$$
r_{t+1}
$$

and reconstructs price forecasts as:

$$
\widehat P_{t+1} = P_t \exp(\widehat r_{t+1})
$$

This avoids the common mistake of training directly on non-stationary price levels without careful treatment.

## 3. ARIMA model

An ARIMA($p,d,q$) model combines:

- AR terms: lagged values;
- differencing: stationarity transformation;
- MA terms: lagged shocks.

For stationary returns, we often use $d=0$:

$$
\begin{aligned}
r_t &= c \\
&\quad + \phi_1 r_{t-1} \\
&\quad + \cdots \\
&\quad + \phi_p r_{t-p} \\
&\quad + \theta_1\varepsilon_{t-1} \\
&\quad + \cdots \\
&\quad + \theta_q\varepsilon_{t-q} \\
&\quad + \varepsilon_t
\end{aligned}
$$

This is ARMA($p,q$), i.e. ARIMA($p,0,q$).

The notebook uses a small AIC grid search over candidate orders.

## 4. LSTM model

An LSTM receives a rolling sequence:

$$
[r_{t-L+1},\dots,r_t]
$$

and predicts:

$$
r_{t+1}
$$

For a multifeature version, the input tensor is:

$$
X \in \mathbb R^{N\times L\times d}
$$

where:

- $N$ is the number of training samples;
- $L$ is the sequence length;
- $d$ is the number of features.

Features in this notebook include:

1. return;
2. absolute return;
3. rolling volatility;
4. rolling mean return;
5. momentum proxy.

Scaling is fit only on the training set.

## 5. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from statsmodels.tsa.arima.model import ARIMA
    STATSMODELS_AVAILABLE = True
except Exception:
    STATSMODELS_AVAILABLE = False

try:
    import tensorflow as tf
    from tensorflow import keras
    TENSORFLOW_AVAILABLE = True
except Exception:
    TENSORFLOW_AVAILABLE = False

try:
    from sklearn.neural_network import MLPRegressor
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

STATSMODELS_AVAILABLE, TENSORFLOW_AVAILABLE, SKLEARN_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class ARIMALSTMConfig:
    n_obs: int = 3_000
    train_fraction: float = 0.60
    validation_fraction: float = 0.20
    annualisation: int = 252
    sequence_length: int = 40
    rolling_vol_window: int = 21
    arima_max_p: int = 3
    arima_max_q: int = 3
    arima_refit_every: int = 50
    arima_rolling_window: int = 900
    lstm_epochs: int = 25
    lstm_batch_size: int = 64
    seed: int = 42


config = ARIMALSTMConfig()
config

## 6. Synthetic financial time series

We simulate a time series with:

1. weak linear autoregression;
2. nonlinear seasonal component;
3. volatility clustering;
4. regime-dependent volatility;
5. heavy-tailed shocks.

This creates a fair testbed:

- ARIMA should capture linear autocorrelation;
- LSTM may exploit nonlinear/rolling features;
- both models must fight noisy returns.

In [ ]:
def simulate_financial_series(config: ARIMALSTMConfig) -> pd.DataFrame:
    rng = np.random.default_rng(config.seed)
    n = config.n_obs

    returns = np.zeros(n)
    latent_vol = np.zeros(n)
    regime = np.zeros(n, dtype=int)

    transition = np.array([
        [0.985, 0.015],
        [0.035, 0.965]
    ])

    regime_vol = np.array([0.008, 0.022])
    regime_mean = np.array([0.00020, -0.00010])

    latent_vol[0] = regime_vol[0]

    for t in range(2, n):
        regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        # Volatility clustering.
        latent_vol[t] = (
            0.94 * latent_vol[t - 1]
            + 0.06 * regime_vol[regime[t]]
            + 0.15 * abs(returns[t - 1])
        )
        latent_vol[t] = np.clip(latent_vol[t], 0.003, 0.080)

        # Linear AR component.
        linear = 0.10 * returns[t - 1] - 0.04 * returns[t - 2]

        # Small nonlinear component.
        nonlinear = 0.00035 * np.sin(2 * np.pi * t / 63) + 0.00020 * np.sign(returns[t - 1]) * min(abs(returns[t - 1]) / 0.02, 1.0)

        # Heavy-tailed shock.
        shock = rng.standard_t(df=7) * np.sqrt((7 - 2) / 7)

        returns[t] = regime_mean[regime[t]] + linear + nonlinear + latent_vol[t] * shock

    dates = pd.bdate_range("2015-01-01", periods=n)
    price = 100 * np.exp(np.cumsum(returns))

    out = pd.DataFrame({
        "timestamp": dates,
        "return": returns,
        "price": price,
        "latent_vol": latent_vol,
        "regime": regime
    })

    out["abs_return"] = out["return"].abs()
    out["squared_return"] = out["return"] ** 2
    out["rolling_vol_21"] = out["return"].rolling(config.rolling_vol_window).std()
    out["rolling_mean_5"] = out["return"].rolling(5).mean()
    out["rolling_mean_21"] = out["return"].rolling(21).mean()
    out["momentum_21"] = np.log(out["price"] / out["price"].shift(21))
    out["target_next_return"] = out["return"].shift(-1)
    out["target_next_price"] = out["price"].shift(-1)

    out = out.dropna().reset_index(drop=True)

    return out


data = simulate_financial_series(config)

data.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data["timestamp"], data["price"])
plt.title("Synthetic Price Series")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(data["timestamp"], data["return"])
plt.title("Synthetic Returns")
plt.xlabel("Date")
plt.ylabel("Return")
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(data["timestamp"], data["regime"], drawstyle="steps-post")
plt.title("Synthetic Hidden Volatility Regime")
plt.xlabel("Date")
plt.ylabel("Regime")
plt.yticks([0, 1])
plt.show()

## 7. Basic diagnostics

We inspect:

1. return distribution;
2. autocorrelation of returns;
3. autocorrelation of squared returns.

The synthetic series should show modest return autocorrelation and stronger volatility persistence.

In [ ]:
def autocorrelation(x, max_lag):
    x = np.asarray(x, dtype=float)
    x = x - np.mean(x)
    denom = np.sum(x ** 2)

    rows = []
    for lag in range(1, max_lag + 1):
        rows.append({
            "lag": lag,
            "autocorrelation": float(np.sum(x[lag:] * x[:-lag]) / denom)
        })

    return pd.DataFrame(rows)


acf_ret = autocorrelation(data["return"], 40).assign(series="return")
acf_sq = autocorrelation(data["squared_return"], 40).assign(series="squared_return")
acf_abs = autocorrelation(data["abs_return"], 40).assign(series="absolute_return")

acf_table = pd.concat([acf_ret, acf_sq, acf_abs], ignore_index=True)

summary_stats = pd.Series({
    "mean_daily_return": data["return"].mean(),
    "daily_vol": data["return"].std(),
    "annualised_mean_return": data["return"].mean() * config.annualisation,
    "annualised_vol": data["return"].std() * np.sqrt(config.annualisation),
    "skew": data["return"].skew(),
    "excess_kurtosis": data["return"].kurtosis()
})

summary_stats

In [ ]:
plt.figure(figsize=(10, 6))
for name, group in acf_table.groupby("series"):
    plt.plot(group["lag"], group["autocorrelation"], marker="o", label=name)
plt.axhline(0, linestyle="--")
plt.title("Autocorrelation Diagnostics")
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(data["return"], bins=100, density=True)
plt.title("Return Distribution")
plt.xlabel("Return")
plt.ylabel("Density")
plt.show()

## 8. Time-series split

We use three chronological blocks:

1. train;
2. validation;
3. test.

The validation set is used for model selection and neural training monitoring.

The test set is untouched until final evaluation.

In [ ]:
n = len(data)
train_end = int(config.train_fraction * n)
validation_end = int((config.train_fraction + config.validation_fraction) * n)

train_df = data.iloc[:train_end].copy()
val_df = data.iloc[train_end:validation_end].copy()
test_df = data.iloc[validation_end:].copy()

split_metadata = {
    "n_total": int(n),
    "n_train": int(len(train_df)),
    "n_validation": int(len(val_df)),
    "n_test": int(len(test_df)),
    "train_start": str(train_df["timestamp"].iloc[0]),
    "train_end": str(train_df["timestamp"].iloc[-1]),
    "validation_start": str(val_df["timestamp"].iloc[0]),
    "validation_end": str(val_df["timestamp"].iloc[-1]),
    "test_start": str(test_df["timestamp"].iloc[0]),
    "test_end": str(test_df["timestamp"].iloc[-1]),
}

pd.Series(split_metadata)

## 9. Baseline forecasts

A serious benchmark needs dumb baselines.

We use:

### Zero-return forecast

$$
\hat r_{t+1}=0
$$

### Historical mean forecast

$$
\hat r_{t+1}=\bar r_{\text{train}}
$$

### Last return forecast

$$
\hat r_{t+1}=r_t
$$

### Rolling mean forecast

$$
\hat r_{t+1}=\frac{1}{20}\sum_{i=0}^{19} r_{t-i}
$$

In [ ]:
def add_baseline_forecasts(df: pd.DataFrame, train_df: pd.DataFrame) -> pd.DataFrame:
    out = df[["timestamp", "price", "return", "target_next_return", "target_next_price"]].copy()

    train_mean = train_df["return"].mean()

    out["forecast_zero"] = 0.0
    out["forecast_train_mean"] = train_mean
    out["forecast_last_return"] = out["return"]
    out["forecast_rolling_mean_20"] = out["return"].rolling(20).mean().fillna(train_mean)

    return out


val_forecasts = add_baseline_forecasts(val_df, train_df)
test_forecasts = add_baseline_forecasts(test_df, pd.concat([train_df, val_df]))

test_forecasts.head()

## 10. Forecast metrics

We evaluate return forecasts with:

1. MSE;
2. MAE;
3. correlation with realised next return;
4. directional accuracy;
5. simple information ratio of a toy sign strategy.

Directional accuracy:

$$
\operatorname{sign}(\hat r_{t+1})=\operatorname{sign}(r_{t+1})
$$

In [ ]:
def forecast_metrics(df: pd.DataFrame, forecast_col: str, target_col: str = "target_next_return") -> dict:
    y = df[target_col].to_numpy()
    pred = df[forecast_col].to_numpy()

    mask = np.isfinite(y) & np.isfinite(pred)
    y = y[mask]
    pred = pred[mask]

    error = pred - y

    strategy_return = np.sign(pred) * y

    return {
        "forecast": forecast_col,
        "n": int(len(y)),
        "mse": float(np.mean(error ** 2)),
        "mae": float(np.mean(np.abs(error))),
        "forecast_actual_corr": float(np.corrcoef(pred, y)[0, 1]) if np.std(pred) > 0 else np.nan,
        "directional_accuracy": float(np.mean(np.sign(pred) == np.sign(y))),
        "toy_strategy_mean_daily": float(np.mean(strategy_return)),
        "toy_strategy_ann_vol": float(np.std(strategy_return, ddof=1) * np.sqrt(config.annualisation)),
        "toy_strategy_ir": float(np.mean(strategy_return) / np.std(strategy_return, ddof=1) * np.sqrt(config.annualisation)) if np.std(strategy_return, ddof=1) > 0 else np.nan
    }


baseline_cols = ["forecast_zero", "forecast_train_mean", "forecast_last_return", "forecast_rolling_mean_20"]

baseline_test_metrics = pd.DataFrame([
    forecast_metrics(test_forecasts, col)
    for col in baseline_cols
])

baseline_test_metrics

## 11. ARIMA order selection

We use the training and validation data for ARIMA order selection.

Since the target is return, we set:

$$
d=0
$$

and grid search:

$$
p=0,\dots,p_{\max}
$$

$$
q=0,\dots,q_{\max}
$$

The selected order minimises AIC.

If `statsmodels` is unavailable, the notebook falls back to an AR($p$) model implemented by OLS.

In [ ]:
def select_arima_order(train_series, max_p=3, max_q=3):
    if not STATSMODELS_AVAILABLE:
        return {"order": None, "selection_table": pd.DataFrame(), "available": False}

    rows = []
    y = np.asarray(train_series, dtype=float)

    for p in range(max_p + 1):
        for q in range(max_q + 1):
            if p == 0 and q == 0:
                continue

            order = (p, 0, q)

            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    model = ARIMA(y, order=order)
                    fit = model.fit()

                rows.append({
                    "p": p,
                    "d": 0,
                    "q": q,
                    "order": order,
                    "aic": float(fit.aic),
                    "bic": float(fit.bic),
                    "converged": True
                })

            except Exception:
                rows.append({
                    "p": p,
                    "d": 0,
                    "q": q,
                    "order": order,
                    "aic": np.nan,
                    "bic": np.nan,
                    "converged": False
                })

    table = pd.DataFrame(rows).dropna(subset=["aic"]).sort_values("aic")

    if len(table) == 0:
        return {"order": None, "selection_table": pd.DataFrame(rows), "available": False}

    return {"order": tuple(table.iloc[0]["order"]), "selection_table": table, "available": True}


arima_selection = select_arima_order(
    train_df["return"],
    max_p=config.arima_max_p,
    max_q=config.arima_max_q
)

arima_selection["order"], arima_selection["selection_table"].head()

## 12. AR fallback model

If ARIMA is not available, we fit a simple AR($p$) model by OLS.

This is not a full ARIMA replacement, but it preserves the classical linear baseline.

In [ ]:
def fit_ar_ols(series, lag):
    y = np.asarray(series, dtype=float)
    X = []
    target = []

    for t in range(lag, len(y)):
        X.append([1.0] + [y[t - i] for i in range(1, lag + 1)])
        target.append(y[t])

    X = np.asarray(X, dtype=float)
    target = np.asarray(target, dtype=float)

    beta = np.linalg.pinv(X.T @ X) @ X.T @ target

    return beta


def forecast_ar_ols(history, beta):
    lag = len(beta) - 1
    x = np.array([1.0] + [history[-i] for i in range(1, lag + 1)])
    return float(x @ beta)

## 13. Walk-forward ARIMA-style forecasting

A fair time-series forecast uses only past data at each test point.

To avoid excessive runtime, this implementation:

1. uses a rolling history window;
2. refits the ARIMA model periodically;
3. produces one-step forecasts sequentially;
4. appends the realised return to history after each forecast.

This is a practical compromise between full refitting at every step and a static forecast.

In [ ]:
def walk_forward_arima_forecast(
    train_and_val_returns,
    test_returns,
    order,
    config: ARIMALSTMConfig
):
    history = list(np.asarray(train_and_val_returns, dtype=float))
    predictions = []

    if STATSMODELS_AVAILABLE and order is not None:
        current_fit = None
        current_order = order

        for i, actual in enumerate(test_returns):
            if i % config.arima_refit_every == 0 or current_fit is None:
                window = np.asarray(history[-config.arima_rolling_window:], dtype=float)

                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    current_fit = ARIMA(window, order=current_order).fit()

            try:
                pred = float(current_fit.forecast(steps=1)[0])
            except Exception:
                pred = float(np.mean(history[-20:]))

            predictions.append(pred)
            history.append(float(actual))

        method = f"arima_{order}"

    else:
        lag = max(1, config.selected_lag_default)
        beta = fit_ar_ols(history, lag)

        for actual in test_returns:
            pred = forecast_ar_ols(history, beta)
            predictions.append(pred)
            history.append(float(actual))

        method = f"ar_ols_fallback_lag{lag}"

    return np.asarray(predictions), method


train_val_returns = pd.concat([train_df, val_df])["return"].to_numpy()
test_returns = test_df["target_next_return"].to_numpy()

arima_predictions, arima_method_name = walk_forward_arima_forecast(
    train_val_returns,
    test_returns,
    arima_selection["order"],
    config
)

test_forecasts["forecast_arima"] = arima_predictions

arima_method_name, test_forecasts[["timestamp", "forecast_arima", "target_next_return"]].head()

## 14. Sequence-window dataset for LSTM

For each time $t$, the LSTM receives a window:

$$
X_t = [x_{t-L+1},\dots,x_t]
$$

and predicts:

$$
r_{t+1}
$$

Feature columns:

1. return;
2. absolute return;
3. rolling volatility;
4. rolling mean return;
5. momentum.

All features and targets are scaled using training data only.

In [ ]:
sequence_feature_cols = [
    "return",
    "abs_return",
    "rolling_vol_21",
    "rolling_mean_5",
    "rolling_mean_21",
    "momentum_21"
]

def fit_standard_scaler(train_df, cols):
    mean = train_df[cols].mean()
    std = train_df[cols].std(ddof=1).replace(0, 1.0)
    return mean, std


feature_mean, feature_std = fit_standard_scaler(train_df, sequence_feature_cols)
target_mean = train_df["target_next_return"].mean()
target_std = train_df["target_next_return"].std(ddof=1)


def apply_scaling(df, feature_cols, feature_mean, feature_std, target_mean, target_std):
    out = df.copy()

    for col in feature_cols:
        out[f"{col}_scaled"] = (out[col] - feature_mean[col]) / feature_std[col]

    out["target_scaled"] = (out["target_next_return"] - target_mean) / target_std

    return out


scaled_data = apply_scaling(data, sequence_feature_cols, feature_mean, feature_std, target_mean, target_std)

scaled_feature_cols = [f"{c}_scaled" for c in sequence_feature_cols]

scaled_data[["timestamp"] + scaled_feature_cols + ["target_scaled"]].head()

In [ ]:
def make_sequence_dataset(scaled_df, feature_cols, target_col, sequence_length):
    X = []
    y = []
    timestamps = []

    values = scaled_df[feature_cols].to_numpy()
    target = scaled_df[target_col].to_numpy()

    for t in range(sequence_length - 1, len(scaled_df)):
        X.append(values[t - sequence_length + 1:t + 1])
        y.append(target[t])
        timestamps.append(scaled_df["timestamp"].iloc[t])

    return np.asarray(X, dtype=float), np.asarray(y, dtype=float), np.asarray(timestamps)


X_all, y_all, ts_all = make_sequence_dataset(
    scaled_data,
    scaled_feature_cols,
    "target_scaled",
    config.sequence_length
)

sequence_meta = pd.DataFrame({"timestamp": ts_all})

train_end_date = train_df["timestamp"].iloc[-1]
val_end_date = val_df["timestamp"].iloc[-1]

train_mask = sequence_meta["timestamp"] <= train_end_date
val_mask = (sequence_meta["timestamp"] > train_end_date) & (sequence_meta["timestamp"] <= val_end_date)
test_mask = sequence_meta["timestamp"] > val_end_date

X_train_seq, y_train_seq = X_all[train_mask], y_all[train_mask]
X_val_seq, y_val_seq = X_all[val_mask], y_all[val_mask]
X_test_seq, y_test_seq = X_all[test_mask], y_all[test_mask]
ts_test_seq = sequence_meta.loc[test_mask, "timestamp"].to_numpy()

pd.Series({
    "X_train_shape": str(X_train_seq.shape),
    "X_val_shape": str(X_val_seq.shape),
    "X_test_shape": str(X_test_seq.shape)
})

## 15. LSTM training

If TensorFlow is available, we train a compact LSTM.

Design choices:

- small model;
- early stopping;
- validation split is chronological;
- no test leakage;
- MSE loss on scaled return target.

If TensorFlow is unavailable, the notebook uses a small `sklearn` MLP on flattened sequence windows as a neural fallback and labels it clearly as **not an LSTM**.

In [ ]:
def train_lstm_or_fallback(config, X_train, y_train, X_val, y_val):
    if TENSORFLOW_AVAILABLE:
        tf.random.set_seed(config.seed)

        model = keras.Sequential([
            keras.layers.Input(shape=(X_train.shape[1], X_train.shape[2])),
            keras.layers.LSTM(32, return_sequences=False),
            keras.layers.Dropout(0.15),
            keras.layers.Dense(16, activation="relu"),
            keras.layers.Dense(1)
        ])

        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=0.001),
            loss="mse"
        )

        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=5,
                restore_best_weights=True
            )
        ]

        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=config.lstm_epochs,
            batch_size=config.lstm_batch_size,
            verbose=0,
            callbacks=callbacks
        )

        return {
            "model": model,
            "history": pd.DataFrame(history.history),
            "model_type": "tensorflow_lstm",
            "available": True
        }

    if SKLEARN_AVAILABLE:
        X_train_flat = X_train.reshape(X_train.shape[0], -1)
        X_val_flat = X_val.reshape(X_val.shape[0], -1)

        model = MLPRegressor(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            alpha=1e-4,
            learning_rate_init=1e-3,
            max_iter=300,
            early_stopping=True,
            validation_fraction=0.15,
            random_state=config.seed
        )

        model.fit(X_train_flat, y_train)

        # sklearn does not expose same clean loss history for early stopping in all versions.
        history = pd.DataFrame({
            "loss": getattr(model, "loss_curve_", [])
        })

        return {
            "model": model,
            "history": history,
            "model_type": "sklearn_mlp_flattened_sequence_fallback_not_lstm",
            "available": True
        }

    return {
        "model": None,
        "history": pd.DataFrame(),
        "model_type": "no_neural_model_available",
        "available": False
    }


neural_fit = train_lstm_or_fallback(
    config,
    X_train_seq,
    y_train_seq,
    X_val_seq,
    y_val_seq
)

neural_fit["model_type"], neural_fit["available"]

In [ ]:
if not neural_fit["history"].empty:
    plt.figure(figsize=(10, 6))
    for col in neural_fit["history"].columns:
        plt.plot(neural_fit["history"][col], label=col)
    plt.title(f"Neural Training History: {neural_fit['model_type']}")
    plt.xlabel("Epoch / iteration")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()
else:
    print("No neural training history available.")

## 16. Neural test forecasts

We predict scaled returns, then inverse-transform:

$$
\hat r = \hat z \cdot \sigma_{\text{train}} + \mu_{\text{train}}
$$

In [ ]:
def predict_neural(neural_fit, X):
    if not neural_fit["available"]:
        return np.full(X.shape[0], np.nan)

    model = neural_fit["model"]

    if neural_fit["model_type"] == "tensorflow_lstm":
        pred_scaled = model.predict(X, verbose=0).reshape(-1)
    else:
        X_flat = X.reshape(X.shape[0], -1)
        pred_scaled = model.predict(X_flat).reshape(-1)

    return pred_scaled * target_std + target_mean


neural_predictions = predict_neural(neural_fit, X_test_seq)

neural_forecasts = pd.DataFrame({
    "timestamp": ts_test_seq,
    "forecast_neural": neural_predictions
})

test_forecasts = test_forecasts.merge(neural_forecasts, on="timestamp", how="left")

test_forecasts[["timestamp", "forecast_neural", "target_next_return"]].dropna().head()

## 17. Align all test forecasts

The LSTM uses a sequence window, so it starts later than raw baselines.

For fair comparison, we align all methods on timestamps where every relevant forecast exists.

In [ ]:
all_model_cols = baseline_cols + ["forecast_arima", "forecast_neural"]

aligned_forecasts = test_forecasts.dropna(subset=["target_next_return"] + all_model_cols).copy()

aligned_metrics = pd.DataFrame([
    forecast_metrics(aligned_forecasts, col)
    for col in all_model_cols
]).sort_values("mse")

aligned_metrics

In [ ]:
plt.figure(figsize=(12, 6))
plt.barh(aligned_metrics["forecast"], aligned_metrics["mse"])
plt.title("Forecast MSE Comparison")
plt.xlabel("MSE")
plt.ylabel("Forecast")
plt.show()

plt.figure(figsize=(12, 6))
plt.barh(aligned_metrics.sort_values("directional_accuracy")["forecast"], aligned_metrics.sort_values("directional_accuracy")["directional_accuracy"])
plt.title("Directional Accuracy Comparison")
plt.xlabel("Directional accuracy")
plt.ylabel("Forecast")
plt.show()

## 18. Forecast visualisation

Financial return forecasts are usually tiny.

A model can have good MSE but still look visually unimpressive because return noise is large.

We plot a small test sample.

In [ ]:
plot_sample = aligned_forecasts.iloc[:250]

plt.figure(figsize=(12, 6))
plt.plot(plot_sample["timestamp"], plot_sample["target_next_return"], label="Actual next return", alpha=0.7)
plt.plot(plot_sample["timestamp"], plot_sample["forecast_arima"], label="ARIMA forecast", alpha=0.8)
plt.plot(plot_sample["timestamp"], plot_sample["forecast_neural"], label="Neural forecast", alpha=0.8)
plt.axhline(0, linestyle="--")
plt.title("Return Forecasts: Sample of Test Period")
plt.xlabel("Date")
plt.ylabel("Return")
plt.legend()
plt.show()

## 19. Price reconstruction

Although we forecast returns, we can reconstruct one-step-ahead price forecasts:

$$
\widehat P_{t+1}=P_t e^{\widehat r_{t+1}}
$$

This is useful for plotting, but metrics should usually be assessed on returns or trading/risk outcomes.

In [ ]:
for col in all_model_cols:
    price_col = col.replace("forecast_", "price_forecast_")
    aligned_forecasts[price_col] = aligned_forecasts["price"] * np.exp(aligned_forecasts[col])

price_cols_forecast = [c.replace("forecast_", "price_forecast_") for c in all_model_cols]

price_error_rows = []

for col in price_cols_forecast:
    error = aligned_forecasts[col] - aligned_forecasts["target_next_price"]
    price_error_rows.append({
        "model": col,
        "price_rmse": float(np.sqrt(np.mean(error ** 2))),
        "price_mae": float(np.mean(np.abs(error)))
    })

price_metrics = pd.DataFrame(price_error_rows).sort_values("price_rmse")

price_metrics

In [ ]:
plot_sample = aligned_forecasts.iloc[:250]

plt.figure(figsize=(12, 6))
plt.plot(plot_sample["timestamp"], plot_sample["target_next_price"], label="Actual next price", linewidth=2)
plt.plot(plot_sample["timestamp"], plot_sample["price_forecast_arima"], label="ARIMA price forecast", alpha=0.8)
plt.plot(plot_sample["timestamp"], plot_sample["price_forecast_neural"], label="Neural price forecast", alpha=0.8)
plt.title("One-Step Price Forecast Reconstruction")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.show()

## 20. Diebold-Mariano-style paired loss comparison

A simple paired comparison uses loss differences:

$$
d_t = L(e_{1,t}) - L(e_{2,t})
$$

If:

$$
\bar d < 0
$$

then model 1 has lower average loss.

A full Diebold-Mariano test requires careful autocorrelation adjustment for multi-step forecasts. Here we use a simple one-step paired t-style diagnostic as an educational approximation.

In [ ]:
def paired_loss_comparison(df, model_a, model_b, target_col="target_next_return", loss="squared"):
    y = df[target_col].to_numpy()
    pred_a = df[model_a].to_numpy()
    pred_b = df[model_b].to_numpy()

    err_a = pred_a - y
    err_b = pred_b - y

    if loss == "squared":
        loss_a = err_a ** 2
        loss_b = err_b ** 2
    elif loss == "absolute":
        loss_a = np.abs(err_a)
        loss_b = np.abs(err_b)
    else:
        raise ValueError("loss must be squared or absolute.")

    d = loss_a - loss_b
    mean_d = float(np.mean(d))
    se_d = float(np.std(d, ddof=1) / np.sqrt(len(d)))

    t_stat = mean_d / se_d if se_d > 0 else np.nan

    return {
        "model_a": model_a,
        "model_b": model_b,
        "loss": loss,
        "mean_loss_diff_a_minus_b": mean_d,
        "standard_error": se_d,
        "t_style_stat": t_stat,
        "interpretation": "negative means model_a lower loss"
    }


paired_comparisons = pd.DataFrame([
    paired_loss_comparison(aligned_forecasts, "forecast_arima", "forecast_neural", loss="squared"),
    paired_loss_comparison(aligned_forecasts, "forecast_arima", "forecast_zero", loss="squared"),
    paired_loss_comparison(aligned_forecasts, "forecast_neural", "forecast_zero", loss="squared"),
    paired_loss_comparison(aligned_forecasts, "forecast_arima", "forecast_neural", loss="absolute"),
])

paired_comparisons

## 21. Toy trading signal

We convert forecasts into a toy directional strategy:

$$
w_t = \operatorname{clip}\Big(\frac{\hat r_{t+1}}{\tau}, -1, 1\Big)
$$

where $\tau$ is a volatility scaling constant.

Strategy return:

$$
R_{strat,t+1}=w_t r_{t+1}-cost\cdot|w_t-w_{t-1}|
$$

This is not an alpha claim.

It is a diagnostic showing whether forecast signs and magnitudes have any economic usefulness after simple costs.

In [ ]:
def toy_strategy_from_forecast(df, forecast_col, cost_per_turnover=0.00005):
    out = df[["timestamp", "target_next_return", forecast_col]].copy()

    scale = out[forecast_col].std()
    scale = scale if scale > 1e-12 else 1e-4

    out["weight"] = np.clip(out[forecast_col] / (2.0 * scale), -1.0, 1.0)
    out["turnover"] = out["weight"].diff().abs().fillna(out["weight"].abs())
    out["strategy_return"] = out["weight"] * out["target_next_return"] - cost_per_turnover * out["turnover"]
    out["cum_strategy"] = np.exp(out["strategy_return"].cumsum())

    return out


strategy_frames = []
strategy_summaries = []

for col in ["forecast_arima", "forecast_neural", "forecast_zero", "forecast_rolling_mean_20"]:
    strat = toy_strategy_from_forecast(aligned_forecasts, col)
    strat["model"] = col
    strategy_frames.append(strat)

    strategy_summaries.append({
        "model": col,
        "mean_daily_return": strat["strategy_return"].mean(),
        "annualised_return": strat["strategy_return"].mean() * config.annualisation,
        "annualised_vol": strat["strategy_return"].std(ddof=1) * np.sqrt(config.annualisation),
        "information_ratio": strat["strategy_return"].mean() / strat["strategy_return"].std(ddof=1) * np.sqrt(config.annualisation) if strat["strategy_return"].std(ddof=1) > 0 else np.nan,
        "average_abs_weight": strat["weight"].abs().mean(),
        "average_turnover": strat["turnover"].mean(),
        "total_log_return": strat["strategy_return"].sum()
    })

strategy_df = pd.concat(strategy_frames, ignore_index=True)
strategy_summary = pd.DataFrame(strategy_summaries).sort_values("information_ratio", ascending=False)

strategy_summary

In [ ]:
plt.figure(figsize=(12, 6))
for model, group in strategy_df.groupby("model"):
    plt.plot(group["timestamp"], group["cum_strategy"], label=model)
plt.title("Toy Strategy Cumulative Growth")
plt.xlabel("Date")
plt.ylabel("Cumulative growth proxy")
plt.legend()
plt.show()

## 22. Error by volatility regime

Deep models can sometimes look good overall but fail during high-volatility regimes.

We bucket test observations by realised volatility and compare forecast errors.

In [ ]:
aligned_forecasts["realized_abs_return"] = aligned_forecasts["target_next_return"].abs()
aligned_forecasts["vol_regime"] = pd.qcut(
    aligned_forecasts["realized_abs_return"],
    q=4,
    labels=["low", "mid_low", "mid_high", "high"]
)

regime_rows = []

for regime, group in aligned_forecasts.groupby("vol_regime", observed=True):
    for col in ["forecast_arima", "forecast_neural", "forecast_zero"]:
        err = group[col] - group["target_next_return"]
        regime_rows.append({
            "vol_regime": str(regime),
            "forecast": col,
            "n": len(group),
            "mse": float(np.mean(err ** 2)),
            "mae": float(np.mean(np.abs(err))),
            "directional_accuracy": float(np.mean(np.sign(group[col]) == np.sign(group["target_next_return"])))
        })

regime_error = pd.DataFrame(regime_rows)

regime_error

In [ ]:
plt.figure(figsize=(10, 6))
for col, group in regime_error.groupby("forecast"):
    plt.plot(group["vol_regime"], group["mse"], marker="o", label=col)
plt.title("Forecast MSE by Realised Volatility Regime")
plt.xlabel("Volatility regime")
plt.ylabel("MSE")
plt.legend()
plt.show()

## 23. Leakage and benchmark checklist

A fair ARIMA vs LSTM benchmark requires:

1. **Chronological split**  
   No random train/test split.

2. **Train-only scaling**  
   Feature means and standard deviations fit on train only.

3. **Stationary target**  
   Forecast returns, differences, or otherwise justified transformations.

4. **Baselines**  
   Include naïve models.

5. **Validation set**  
   Tune neural models without using the test set.

6. **Same forecast horizon**  
   Compare one-step with one-step.

7. **Aligned timestamps**  
   Evaluate all models on the same test observations.

8. **Cost-aware economics**  
   Forecast MSE is not the same as tradable alpha.

9. **Regime diagnostics**  
   Check high-volatility periods separately.

10. **Model availability**  
   Report whether LSTM was actually trained or fallback was used.

## 24. Saving outputs

The notebook saves:

1. synthetic time series;
2. autocorrelation diagnostics;
3. ARIMA order selection;
4. aligned forecast table;
5. return forecast metrics;
6. price forecast metrics;
7. paired loss comparison;
8. toy strategy results;
9. regime error diagnostics;
10. model availability and manifest.

In [ ]:
output_dir = Path("data/processed/arima_vs_lstm_benchmark_v1")
output_dir.mkdir(parents=True, exist_ok=True)

data_path = output_dir / "synthetic_time_series.csv"
acf_path = output_dir / "autocorrelation_diagnostics.csv"
summary_path = output_dir / "summary_statistics.csv"
split_path = output_dir / "split_metadata.json"
arima_selection_path = output_dir / "arima_order_selection.csv"
forecasts_path = output_dir / "aligned_test_forecasts.csv"
metrics_path = output_dir / "forecast_metrics.csv"
price_metrics_path = output_dir / "price_forecast_metrics.csv"
paired_path = output_dir / "paired_loss_comparison.csv"
strategy_path = output_dir / "toy_strategy_returns.csv"
strategy_summary_path = output_dir / "toy_strategy_summary.csv"
regime_error_path = output_dir / "forecast_error_by_vol_regime.csv"
neural_history_path = output_dir / "neural_training_history.csv"
manifest_path = output_dir / "manifest.json"

data.to_csv(data_path, index=False)
acf_table.to_csv(acf_path, index=False)
summary_stats.to_frame("value").to_csv(summary_path)
split_path.write_text(json.dumps(split_metadata, indent=2, default=str))

if arima_selection["selection_table"] is not None and not arima_selection["selection_table"].empty:
    arima_selection["selection_table"].to_csv(arima_selection_path, index=False)
else:
    pd.DataFrame([{"note": "statsmodels unavailable or ARIMA selection failed"}]).to_csv(arima_selection_path, index=False)

aligned_forecasts.to_csv(forecasts_path, index=False)
aligned_metrics.to_csv(metrics_path, index=False)
price_metrics.to_csv(price_metrics_path, index=False)
paired_comparisons.to_csv(paired_path, index=False)
strategy_df.to_csv(strategy_path, index=False)
strategy_summary.to_csv(strategy_summary_path, index=False)
regime_error.to_csv(regime_error_path, index=False)
neural_fit["history"].to_csv(neural_history_path, index=False)

manifest = {
    "dataset_name": "arima_vs_lstm_benchmark_outputs",
    "schema_version": "arima_vs_lstm_benchmark_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "statsmodels_available": STATSMODELS_AVAILABLE,
    "tensorflow_available": TENSORFLOW_AVAILABLE,
    "sklearn_available": SKLEARN_AVAILABLE,
    "arima_method_name": arima_method_name,
    "selected_arima_order": str(arima_selection["order"]),
    "neural_model_type": neural_fit["model_type"],
    "best_model_by_mse": aligned_metrics.sort_values("mse").iloc[0].to_dict(),
    "best_model_by_directional_accuracy": aligned_metrics.sort_values("directional_accuracy", ascending=False).iloc[0].to_dict(),
    "best_strategy_by_ir": strategy_summary.sort_values("information_ratio", ascending=False).iloc[0].to_dict(),
    "notes": [
        "Synthetic series contains weak linear autocorrelation, nonlinear components, heavy-tailed shocks, and regime-dependent volatility.",
        "Forecast target is next return, not raw price level.",
        "Feature scaling is fit on training data only.",
        "ARIMA order selection uses AIC when statsmodels is available.",
        "LSTM is trained only if TensorFlow is available; otherwise sklearn MLP fallback may be used and is clearly labelled as not LSTM.",
        "Aligned metrics compare all available forecasts on the same timestamps.",
        "Toy strategy is diagnostic only, not an alpha claim."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

data_path, forecasts_path, metrics_path, strategy_summary_path, manifest_path

## 25. Limitations

### 25.1 Synthetic data

The benchmark uses synthetic data.

Real financial data has regime changes, corporate actions, market microstructure effects, holidays, and survivorship issues.

### 25.2 LSTM availability

The notebook trains a real LSTM only if TensorFlow is installed.

Otherwise it uses a clearly labelled sequence-MLP fallback.

### 25.3 Return predictability is weak

Even if the synthetic data contains weak signal, real return forecasting is much harder.

Small improvements in MSE may not translate into tradable alpha.

### 25.4 ARIMA is not the only classical model

Other baselines include:

- ARMAX;
- VAR;
- GARCH;
- state-space models;
- Kalman filters;
- tree models.

### 25.5 LSTM is not automatically better

A neural model can overfit noise or exploit leakage.

It must beat simple baselines out of sample.

### 25.6 No hyperparameter search

The LSTM architecture is intentionally small.

A full benchmark would tune sequence length, hidden size, dropout, learning rate, and features.

### 25.7 No transaction-cost realism

The toy strategy uses a simple turnover cost.

Real execution requires spread, slippage, market impact, latency, borrow/margin, and capacity analysis.

## 26. Research frontier and extensions

Important extensions include:

1. **ARIMA with exogenous variables**  
   ARIMAX using volatility, regime, macro, or cross-asset features.

2. **State-space models**  
   Kalman filtering and dynamic linear models.

3. **Temporal convolutional networks**  
   Often stronger and easier to train than LSTMs.

4. **Transformers for time series**  
   Useful but highly prone to overfitting in low signal-to-noise finance data.

5. **Hybrid models**  
   ARIMA residuals modelled with LSTM, or neural model with AR residual correction.

6. **Probabilistic forecasts**  
   Predict distributions, not only point forecasts.

7. **Quantile loss**  
   Forecast downside risk directly.

8. **Walk-forward hyperparameter tuning**  
   Refit and tune models in a realistic live protocol.

9. **Cross-asset sequence learning**  
   Multivariate LSTM/TCN using futures, FX, rates, and volatility features.

10. **Chinese futures extension**  
   Include night-session effects, contract rolls, price limits, and product-specific seasonality.

## 27. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_07_cross_sectional_alpha_features**  
   Turning time-series forecasts into cross-sectional features.

2. **03_08_tree_models_for_return_prediction**  
   Compare neural sequence models with gradient boosting.

3. **03_09_information_coefficient_analysis**  
   Evaluate forecast quality as a signal.

4. **03_10_walk_forward_model_validation**  
   Proper rolling model validation framework.

5. **05_01_vectorized_backtest_engine**  
   Backtest forecast-driven signals with costs.

6. **06_12_gpu_monte_carlo_engine**  
   Larger-scale neural and simulation infrastructure.

7. **07_01_multi_asset_cta_research_pipeline**  
   Use ARIMA/LSTM forecasts as candidate timing features.

## 28. Summary

This notebook implemented an ARIMA vs LSTM benchmark.

It showed how to:

1. simulate a noisy financial time series;
2. choose a stationary return target;
3. split data chronologically into train, validation, and test;
4. build naïve baselines;
5. select ARIMA orders using AIC;
6. run walk-forward ARIMA-style forecasting;
7. create sequence windows for neural models;
8. train an LSTM when TensorFlow is available;
9. use a clearly labelled fallback when TensorFlow is unavailable;
10. align forecasts on common timestamps;
11. compare MSE, MAE, correlation, and directional accuracy;
12. reconstruct price forecasts;
13. run paired loss comparisons;
14. test a toy trading signal;
15. diagnose errors by volatility regime;
16. save outputs and metadata.

The key computational takeaway:

> The benchmark protocol matters more than model glamour. A leakage-free baseline can beat a poorly validated neural network.

The key financial takeaway:

> Forecast accuracy is not alpha unless it survives time splits, baselines, costs, and regime stress.

## 29. Further reading

- Box, Jenkins, Reinsel, and Ljung, *Time Series Analysis: Forecasting and Control.*
- Hamilton, *Time Series Analysis.*
- Hyndman and Athanasopoulos, *Forecasting: Principles and Practice.*
- Hochreiter and Schmidhuber, "Long Short-Term Memory."
- Literature on walk-forward validation, time-series cross-validation, deep learning for financial forecasting, and forecast comparison tests.